This notebook contains the main model for the project. 
The idea is to create a machine learning model for a time series, taht predicts the temperature for a few days. 

The data is the measurements of the temperature and the incoming solar irradiance in an hourly-mean from the weather station at the UniSport in Cologne. The measurement periods were March, April, May of the years, 2024, 2025 and 2026. It was preprocessed from 10min mean to one hour mean and to have only the temperature and the radiation data. 


In [19]:
# Import the needed modules

import pandas as pd 
import numpy as np 
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
import pickle 
import torch.nn as nn
import torch.optim as optim

In [12]:
# Read in the preprocessed data

df=pd.read_csv('data_processed.csv', parse_dates=['TIMESTAMP'])

df

,TIMESTAMP,AirTC_2_Avg,SWUpper_Avg
0,2024-03-01 00:00:00,7.202200,0.000000
1,2024-03-01 01:00:00,6.842833,0.000000
2,2024-03-01 02:00:00,6.663333,0.000000
3,2024-03-01 03:00:00,7.020667,0.000000
4,2024-03-01 04:00:00,7.263167,0.000000
...,...,...,...
6621,2026-05-31 19:00:00,20.578333,4.426333
6622,2026-05-31 20:00:00,19.611667,0.000000
6623,2026-05-31 21:00:00,18.628333,0.000000
6624,2026-05-31 22:00:00,17.856667,0.000000


In [13]:
#normalize data

scale=MinMaxScaler()

df[['AirTC_2_Avg', 'SWUpper_Avg']]=scale.fit_transform(df[['AirTC_2_Avg', 'SWUpper_Avg']])

with open ('scale.pkl', 'wb') as f:
    pickle.dump(scale, f)
 

print(df[['AirTC_2_Avg', 'SWUpper_Avg']].describe())

       AirTC_2_Avg  SWUpper_Avg
count  6626.000000  6626.000000
mean      0.408224     0.179819
std       0.166296     0.253787
min       0.000000     0.000000
25%       0.286057     0.000000
50%       0.395750     0.029450
75%       0.515576     0.302490
max       1.000000     1.000000


In [14]:
#split the data in train und test data

train=df[df['TIMESTAMP'].dt.year.isin([2024, 2025])]
test= df[df['TIMESTAMP'].dt.year==2026]

print(len(train))
print(len(test))

4418
2208


In [ ]:
#sliding window

window_size=24  #size of the window
 
def sequences(data):
    X, y=[],[]
    values=data[['AirTC_2_Avg','SWUpper_Avg']].values

    for i in range(len(values)-window_size):
        X.append(values[i:i+window_size])
        y.append(values[i+window_size][0])

    return np.array(X), np.array(y)

X_train, y_train= sequences(train)
X_test, y_test=sequences(test)

print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)



(4394, 24, 2)
(4394,)
(2184, 24, 2)
(2184,)


In [16]:
class WeatherData(Dataset):
    def __init__(self, X, y):
        self.X=torch.tensor(X,dtype=torch.float32)
        self.y=torch.tensor(y,dtype=torch.float32)

    def __len__(self):
        return(len(self.X))
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]
    

train_dataset=WeatherData(X_train, y_train)
test_dataset=WeatherData(X_test, y_test)

batch_size=32

train_loader=DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader=DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(len(test_loader))
print(len(train_loader))


69
138


In [ ]:
class FFN(nn.Module):
    def __init__(self):
        super (FFN, self).__init__()

        self.model=nn.Sequential(nn.Linear(24*2,64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 1))

    
    def forward(self,x):
        x=x.view(x.size(0),-1)
        return self.model(x).squeeze(1)
    

model=FFN()
print(model)


FFN(
  (model): Sequential(
    (0): Linear(in_features=48, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=32, bias=True)
    (3): ReLU()
    (4): Linear(in_features=32, out_features=1, bias=True)
  )
)
Parameter: 5249


In [ ]:
crit=nn.MSELoss() #mean squared-error--> how 'wring' is model?

optimizer=optim.Adam(model.parameters(), lr=0.001)  #learning rate


run=50 #

for r in range(run):
    model.train()
    total_loss=0

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        predicitions=model(X_batch)
        loss=crit(predicitions, y_batch)
        loss.backward()
        optimizer.step()
        total_loss+=loss.item()

    avg_loss=total_loss/len(train_loader)

    if (r+1)%10==0:
        print((r+1)/run, avg_loss)


0.2 0.000572410795122277
0.4 0.00046073412427769813
0.6 0.00044679469061995167
0.8 0.00041590994310420194
1.0 0.000371703207286149
